In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [2]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

## LER OS RETORNOS, SCORE e SELIC

In [3]:
# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    nome = f'df_ativos_{ano_usado}.csv'
    # caminho = Path('retornos_ativos_otimizacao') / nome
    dfs[ano_usado] = pd.read_csv('../dados_retornos/otimizacao/ativos/'+nome)
    dfs[ano_usado]['date'] = pd.to_datetime(dfs[ano_usado]['date'])
    dfs[ano_usado] = dfs[ano_usado].set_index('date')
    dfs[ano_usado] = dfs[ano_usado].drop(columns=dfs[ano_usado].columns[(dfs[ano_usado] == 0).all()])
    print(ano,'-',len(dfs[ano_usado].columns))
    

2025-12-31 - 100
2024-12-31 - 99
2023-12-31 - 99
2022-12-31 - 99
2021-12-31 - 97
2020-12-31 - 84
2019-12-31 - 77
2018-12-31 - 76
2017-12-31 - 73
2016-12-31 - 69
2015-12-31 - 70


In [4]:
dfs['2020']

,ABEV3,ALOS3,ALPA4,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,...,TIMS3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVA3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,
2020-10-01,0.012759,0.046374,0.043979,0.083557,-0.001979,0.020059,-0.008726,0.005739,0.001665,0.004636,...,0.000765,0.015129,0.012974,0.003988,-0.004229,0.042184,0.016878,-0.002494,0.018265,0.021525
2020-10-02,-0.013386,-0.019742,-0.015797,-0.008490,-0.019498,-0.009020,-0.023290,-0.002686,0.004837,0.008158,...,-0.007651,-0.016721,-0.019979,-0.004965,-0.009174,-0.030476,-0.007054,-0.006591,0.002242,-0.031786
2020-10-05,0.022346,0.008631,-0.003567,0.010346,0.054601,0.024759,0.046752,0.017503,0.022075,0.014257,...,0.014649,-0.001849,0.012023,0.054890,0.021776,0.010314,-0.001672,-0.004804,0.046681,0.022501
2020-10-06,0.041374,0.016707,0.018410,-0.027895,-0.000959,-0.016344,-0.035695,-0.006616,-0.005939,-0.003514,...,0.001520,-0.011111,-0.008265,-0.023652,-0.016278,-0.017015,-0.002512,0.014482,0.018524,-0.010462
2020-10-07,0.014243,0.010821,0.011298,-0.004722,0.001919,0.003792,-0.000930,-0.007326,-0.013036,-0.010579,...,-0.002276,0.009363,-0.030208,0.026163,0.026442,-0.010880,-0.007134,-0.013143,0.023643,-0.006562
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-03-25,0.012838,0.013222,0.045573,0.053107,0.049551,0.010857,0.018346,0.014413,0.020345,0.022274,...,0.012698,0.013290,0.035971,-0.003654,-0.007622,0.046031,0.016729,0.004445,0.007799,0.038188
2021-03-26,0.010674,-0.009321,0.012138,-0.020386,-0.003541,0.009595,0.004277,0.003383,0.008236,0.006762,...,0.006270,-0.001063,0.037698,0.013447,0.033427,-0.008532,-0.025092,0.003097,0.023498,-0.023080
2021-03-29,0.009901,-0.007151,-0.023167,-0.016429,-0.018656,-0.019716,-0.005483,0.004720,0.007308,0.004104,...,-0.010125,-0.011001,0.004302,0.005428,0.025646,-0.018116,-0.046414,0.003971,-0.008337,-0.014027


In [5]:
## SELIC para Sigma e excesso
selic_d = pd.read_excel('../score-raspagem/selic/selic.xlsx').set_index('Date')

## EXCESSO DOS ANOS e SIGMA (MAtriz de covariancia)

##### EXCESSO PARA TODOS

In [6]:
selic_d['selic']

Date
2015-01-02    0.043739
2015-01-05    0.043739
2015-01-06    0.043739
2015-01-07    0.043739
2015-01-08    0.043739
                ...   
2026-05-26    0.053400
2026-05-27    0.053400
2026-05-28    0.053400
2026-05-29    0.053400
2026-06-01    0.053400
Name: selic, Length: 2862, dtype: float64

In [7]:
dict_excesso = {}
for an in anos:
    ano = an.split("-")[0]
    try:
        print(f"=============== \n EXCESSO {ano}\n ============")
        print("Atualização, Ano: ",ano)
        df = dfs[ano]
        # df_f = pd.DataFrame(eval(df))
        df_f = df.copy()
        slc = selic_d[selic_d.index.isin(df_f.index)]
        slc['valor_diario'] = slc['selic']/100

        print(f"Tamanho DF de {ano}: ", len(df_f))
        print(f"Tamanho Selic: ", len(slc['valor_diario']))
        ano = int(ano)
        dict_excesso[ano] = df_f.sub(slc['valor_diario'],axis=0)
        print("tamanho final do Excesso: ",len(dict_excesso[ano]))
    except Exception as e:
        print(e)
        print("ERror")

 EXCESSO 2025
Atualização, Ano:  2025
Tamanho DF de 2025:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 EXCESSO 2024
Atualização, Ano:  2024
Tamanho DF de 2024:  122
Tamanho Selic:  122
tamanho final do Excesso:  122
 EXCESSO 2023
Atualização, Ano:  2023
Tamanho DF de 2023:  121
Tamanho Selic:  121
tamanho final do Excesso:  121
 EXCESSO 2022
Atualização, Ano:  2022
Tamanho DF de 2022:  124
Tamanho Selic:  124
tamanho final do Excesso:  124
 EXCESSO 2021
Atualização, Ano:  2021
Tamanho DF de 2021:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 EXCESSO 2020
Atualização, Ano:  2020
Tamanho DF de 2020:  120
Tamanho Selic:  120
tamanho final do Excesso:  120
 EXCESSO 2019
Atualização, Ano:  2019
Tamanho DF de 2019:  123
Tamanho Selic:  123
tamanho final do Excesso:  123
 EXCESSO 2018
Atualização, Ano:  2018
Tamanho DF de 2018:  119
Tamanho Selic:  119
tamanho final do Excesso:  119
 EXCESSO 2017
Atualização, Ano:  2017
Tamanho DF de 2017:  126
Tamanho Selic:  122
taman

## Hiperparâmetros

In [8]:
vb_cardinalidade_max = 10
vb_cardinalidade_min = 10
vb_peso_maximo = 0.20
vb_peso_minimo = 0.02
vb_theta = 0.5

### Fazendo otimização ano a ano e comparando com o proximo ano

In [9]:
anos = ['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']
print(anos)

['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']


In [10]:
y = []

carteiras_anuais = {}

melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])

print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)

for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass
    
    # Retornos só para filtrar colunas
    df_usado = dfs[str(ano)]
    retorno_usado = df_usado.copy()    
        #SCORES MF e PIO------------
    score_mf_todos_anos = pd.read_excel(f'../score-raspagem/magic_formula/resultado/{int(ano)}.xlsx')
    score_mf = score_mf_todos_anos[['Unnamed: 0', 'magic_formula']].set_index('Unnamed: 0').T
    
    score_pio_todos_anos = pd.read_excel(f'../score-raspagem/piotroski/resultado/{int(ano)}.xlsx')
    score_pio = score_pio_todos_anos[['Unnamed: 0', 'p_norm']].set_index('Unnamed: 0').T
    
    ativos = retorno_usado.columns.intersection(score_pio.columns).intersection(score_mf.columns)

    score_usado_mf = score_mf[ativos].copy()
    score_usado_pio = score_pio[ativos].copy()

    # print(score_usado)
    print("Score Atualizado")

    ano_um = int(ano)
    # df_usado = f'df_ativos_{ano_um}'
    # retorno_usado = pd.DataFrame(eval(df_usado))
    # df_usado = dfs[str(ano_um)][ativos_finais]
    # retorno_usado = df_usado.copy()
    # retorno_usado = retorno_usado[lista_ativos_finais]
    # print("Retornos atualizados")

    # sigma_usado = retorno_usado.cov()
    # print("Sigmas Atualizados")
    print("-----")
    

    # if df_usado == 'df_ativos_2015':
    #     continue
    # else:
    print(f"{an} TAMANHOS: Retorno {len(ativos)}, Score PIO {len(score_usado_pio.columns)}, Score MF {len(score_usado_mf.columns)}")
    print("# ------ CRIAÇÃO DO MODELO")
    print("## Carteira criada para o ano: ",int(ano)+1)

    # # print("## UTILIZANDO SCORE DO ANO DE: ",ano)
    # print("## UTILIZANDO DADOS DE RETORNO DE: ",str(ano_um))
    # # print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    # print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)

    model = pyo.ConcreteModel()

    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = ativos)
    model.ativos = pyo.RangeSet(0, len(ativos)-1)
    # model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    # model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    # model.theta = pyo.Param(initialize=vb_theta)
    model.score_p = pyo.Param(model.ativos, initialize=lambda model,a: score_usado_pio.iloc[0,a])
    model.score_m = pyo.Param(model.ativos, initialize=lambda model,a: score_usado_mf.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos, bounds=(0,1))
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    # model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    # model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    # model.s = pyo.Param(initialize = 2, mutable=True)
    # model.r = pyo.Var(within=pyo.NonNegativeReals)
    
    for i in range(3):
        setattr(model,f'obj{i}_mais',pyo.Var(domain=NonNegativeReals)) #bounds=(0,0.1),
        setattr(model,f'obj{i}_menos',pyo.Var(domain=NonNegativeReals))
    #-------------------------------------- FUNÇÕES

    #=============================
    # Função Objetivo
    #=============================

    def func_objetivo_1(model):
       
        return model.obj1_menos + model.obj2_menos
    model.objetivo = pyo.Objective(rule=func_objetivo_1, sense=pyo.minimize)

    #=============================
    # RESTRIÇÕES
    #=============================

    # # OBJ 1 2 3 - Rest 1 2 3
    # def retorno_restr_rule(model,diad):
    #     return sum(model.x[a]*model.retornos_ativos[dia,a] for a in model.ativos for dia in model.dias) + model.obj0_menos - model.obj0_mais >= model.retornos_ibov[diad]
    # model.retorno_restr = pyo.Constraint(model.dias,rule=retorno_restr_rule)

    def piotroski_restr_rule(model):
        return sum(model.x[a]*model.score_p[a] for a in model.ativos) + model.obj1_menos - model.obj1_mais >= 0.9
    model.piotroski_rest = pyo.Constraint(rule=piotroski_restr_rule)

    def mf_restr_rule(model):
        return sum(model.x[a]*model.score_m[a] for a in model.ativos) + model.obj2_menos - model.obj2_mais >= 0.9
    model.mf_restr = pyo.Constraint(rule=mf_restr_rule)

    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)

    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)

    # #peso minimo por acao
    # def rule_peso_minimo(model, a):
    #     return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    # model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)

    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)


    # def cardinalidade_min(model):
    #     return sum(
    #         model.y[a] for a in model.ativos
    #         ) >= model.cardinalidade_valor_min
    # model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)

    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)


    # NOTEBOOOK
    # opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # PC
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
    res = opt.solve(model,tee=False)


    melhor_pesos = {list(model.nome_ativos.data())[a]: pyo.value(model.x[a]) for a in model.ativos}
    pio = {a: sum(pyo.value(model.x[a])*pyo.value(model.score_p[a]) for a in model.ativos)}
    mf = {a: sum(pyo.value(model.x[a])*pyo.value(model.score_m[a]) for a in model.ativos)}
    carteiras_anuais[int(ano_um)+1] = {
        'pesos':  melhor_pesos,
        # 'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano_um} -> {melhor_pesos}")
    print(f"{ano_um} Piotroski -> {pio}")
    # print(f"Valores obj Menos: {pyo.value(model.obj0_menos), pyo.value(model.obj1_menos), pyo.value(model.obj2_menos)}")
    print(f"Valores obj Menos: {pyo.value(model.obj1_menos), pyo.value(model.obj1_mais)}")
    print(f"{ano_um} Magic Formula -> {mf}")
    # print(f"Valores obj Menos: {pyo.value(model.obj0_menos), pyo.value(model.obj1_menos), pyo.value(model.obj2_menos)}")
    print(f"Valores obj Menos: {pyo.value(model.obj2_menos), pyo.value(model.obj2_mais)}")
    print("+-="*30)
    if 'model' in locals():
        del model
        print('DELETADO DPS DO WHILE')
    else:
        print("Nao consta model")
        pass




=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2015
Nao consta model
Score Atualizado
-----
2015-12-31 TAMANHOS: Retorno 57, Score PIO 57, Score MF 57
# ------ CRIAÇÃO DO MODELO
## Carteira criada para o ano:  2016
2015 -> {'ABEV3': 0.0, 'ALOS3': 0.0, 'ALPA4': 0.0, 'ANIM3': 0.0, 'AXIA3': 0.0, 'AZZA3': 0.0, 'B3SA3': 0.0, 'BEEF3': 0.0, 'CMIG4': 0.0, 'COGN3': 0.0, 'CPFE3': 0.0, 'CPLE3': 0.0, 'CSAN3': 0.0, 'CSMG3': 0.0, 'CSNA3': 0.0, 'CYRE3': 0.0, 'DIRR3': 0.0, 'ECOR3': 0.0, 'EGIE3': 0.0, 'EMBJ3': 0.0, 'ENEV3': 0.0, 'ENGI11': 0.0, 'EQTL3': 0.0, 'EZTC3': 0.0, 'FLRY3': 0.2, 'GGBR4': 0.0, 'GOAU4': 0.0, 'HYPE3': 0.0, 'ISAE4': 0.0, 'ITSA4': 0.0, 'JHSF3': 0.0, 'KLBN11': 0.0, 'LREN3': 0.2, 'MBRF3': 0.0, 'MGLU3': 0.0, 'MOTV3': 0.0,

In [11]:
carteiras_anuais

{2016: {'pesos': {'ABEV3': 0.0,
   'ALOS3': 0.0,
   'ALPA4': 0.0,
   'ANIM3': 0.0,
   'AXIA3': 0.0,
   'AZZA3': 0.0,
   'B3SA3': 0.0,
   'BEEF3': 0.0,
   'CMIG4': 0.0,
   'COGN3': 0.0,
   'CPFE3': 0.0,
   'CPLE3': 0.0,
   'CSAN3': 0.0,
   'CSMG3': 0.0,
   'CSNA3': 0.0,
   'CYRE3': 0.0,
   'DIRR3': 0.0,
   'ECOR3': 0.0,
   'EGIE3': 0.0,
   'EMBJ3': 0.0,
   'ENEV3': 0.0,
   'ENGI11': 0.0,
   'EQTL3': 0.0,
   'EZTC3': 0.0,
   'FLRY3': 0.2,
   'GGBR4': 0.0,
   'GOAU4': 0.0,
   'HYPE3': 0.0,
   'ISAE4': 0.0,
   'ITSA4': 0.0,
   'JHSF3': 0.0,
   'KLBN11': 0.0,
   'LREN3': 0.2,
   'MBRF3': 0.0,
   'MGLU3': 0.0,
   'MOTV3': 0.0,
   'MRVE3': 0.0,
   'MULT3': 0.0,
   'NATU3': 0.0,
   'PETR3': 0.0,
   'PETR4': 0.0,
   'POMO4': 0.0,
   'PRIO3': 0.0,
   'RADL3': 0.2,
   'RENT3': 0.0,
   'SBSP3': 0.0,
   'SLCE3': 0.0,
   'SMTO3': 0.0,
   'SUZB3': 0.0,
   'TAEE11': 0.2,
   'TOTS3': 0.0,
   'UGPA3': 0.2,
   'USIM5': 0.0,
   'VALE3': 0.0,
   'VIVT3': 0.0,
   'WEGE3': 0.0,
   'YDUQ3': 0.0}},
 2017: {'pe

In [12]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])+1
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        if round(v, 4)>0 and v <= vb_peso_maximo*1.10:
            linhas.append({'ano': an, 'ativo': k, 'peso': v})
df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [13]:
df_portfolios[df_portfolios['ano'] == 2025].sort_values('peso', ascending=False)['peso']

46    0.2
47    0.2
48    0.2
49    0.2
50    0.2
Name: peso, dtype: float64

In [14]:
df_portfolios.to_csv('../carteiras/goal_programming/carteiras_goal_programming.csv')
